# CodiEsp NER Training — Google Colab

Trains the Spanish Biomedical BERT NER model on the CodiEsp dataset.

**Before running:** `Runtime → Change runtime type → T4 GPU`

Then **Runtime → Run all** — everything else is automated.

| Step | What happens |
|---|---|
| 1 | GPU check |
| 2 | Clone repo from GitHub |
| 3 | Install dependencies |
| 4 | Download CodiEsp data (cached to Drive) |
| 5 | Build ICD-10 index (cached to Drive) |
| 6 | Train NER model |
| 7 | Save trained model to Google Drive |

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

## 2. Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/tylerklement/medical-coding.git"
REPO_DIR = "/content/medical-coding"

if os.path.isdir(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", sorted(os.listdir('.')))

## 3. Install Dependencies

In [ ]:
!pip install -q "transformers>=4.46" datasets seqeval sentence-transformers faiss-cpu accelerate
print("Dependencies ready.")

## 4. Download CodiEsp Data
Downloads ~11 MB from Zenodo. Cached to Google Drive so future runs skip this step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_CACHE = Path('/content/drive/MyDrive/medical-coding-cache')
DATA_DIR    = Path('data')
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

drive_data = DRIVE_CACHE / 'data'

if (drive_data / 'train' / 'trainX.tsv').exists():
    print("Data found in Drive cache — restoring...")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    shutil.copytree(drive_data, DATA_DIR)
    print("Data restored from Drive cache.")
else:
    print("Downloading CodiEsp from Zenodo (~11 MB)...")
    !python scripts/download_data.py
    print("Caching to Drive for future runs...")
    if drive_data.exists():
        shutil.rmtree(drive_data)
    shutil.copytree(DATA_DIR, drive_data)
    print("Done.")

## 5. Build ICD-10 Index
Downloads ICD-10-CM (~74k codes) and ICD-10-PCS (~79k codes) from CMS. Cached to Drive.

In [ ]:
import json, shutil
from pathlib import Path

drive_cm  = DRIVE_CACHE / 'icd10cm_descriptions.json'
drive_pcs = DRIVE_CACHE / 'icd10pcs_descriptions.json'
local_cm  = DATA_DIR / 'icd10cm_descriptions.json'
local_pcs = DATA_DIR / 'icd10pcs_descriptions.json'

if drive_cm.exists() and drive_cm.stat().st_size > 10_000:
    shutil.copy(drive_cm,  local_cm)
    shutil.copy(drive_pcs, local_pcs)
    print(f"ICD-10 index restored from Drive.")
    print(f"  CM  codes : {len(json.load(open(local_cm))):,}")
    print(f"  PCS codes : {len(json.load(open(local_pcs))):,}")
else:
    print("Building ICD-10 index from CMS...")
    !python scripts/build_icd10_index.py
    shutil.copy(local_cm,  drive_cm)
    shutil.copy(local_pcs, drive_pcs)
    print("ICD-10 index built and cached to Drive.")

## 6. Train
Adjust the config below then run. **T4 estimate: ~25-40 min for 5 epochs on the full dataset.**

In [ ]:
# ── Training config ────────────────────────────────────────────────
EPOCHS      = 5     # early stopping (patience=2) will stop sooner if needed
BATCH_SIZE  = 16    # T4 16 GB VRAM — fits comfortably
GRAD_ACCUM  = 1     # effective batch = BATCH_SIZE * GRAD_ACCUM
LR          = 2e-5
NUM_WORKERS = 2     # Linux/Colab is fine with 2; Mac requires 0
MODEL_OUT   = 'models/ner'

# Set to an int to limit articles for a quick smoke-test (None = full dataset)
MAX_TRAIN = None    # e.g. 50 for a ~3-min smoke-test
MAX_DEV   = None
# ──────────────────────────────────────────────────────────────────

import os
os.makedirs(MODEL_OUT, exist_ok=True)
os.makedirs('outputs', exist_ok=True)

args = (
    f"--no_wandb"
    f" --fp16"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --grad_accum {GRAD_ACCUM}"
    f" --learning_rate {LR}"
    f" --num_workers {NUM_WORKERS}"
    f" --output_dir {MODEL_OUT}"
)
if MAX_TRAIN:
    args += f" --max_train_samples {MAX_TRAIN}"
if MAX_DEV:
    args += f" --max_dev_samples {MAX_DEV}"

print("Command: python train_ner.py", args)
print("=" * 70)
!python train_ner.py {args}

## 7. Save Model to Google Drive
Colab sessions expire — this saves the trained model to Drive so it persists.

In [ ]:
import json, shutil
from pathlib import Path

drive_model = DRIVE_CACHE / 'models' / 'ner'
local_model = Path(MODEL_OUT)

if local_model.exists():
    drive_model.parent.mkdir(parents=True, exist_ok=True)
    if drive_model.exists():
        shutil.rmtree(drive_model)
    shutil.copytree(local_model, drive_model)
    print(f"Model saved to Drive: {drive_model}")
    print(f"Files: {sorted(f.name for f in drive_model.iterdir())}")
else:
    print("Model directory not found — did training complete successfully?")

results_path = Path('outputs/ner_results.json')
if results_path.exists():
    r = json.load(open(results_path))
    print("\nFinal dev-set results:")
    print(f"  Precision : {r.get('eval_precision', 0):.4f}")
    print(f"  Recall    : {r.get('eval_recall',    0):.4f}")
    print(f"  F1        : {r.get('eval_f1',        0):.4f}")
    print(f"  Loss      : {r.get('eval_loss',      0):.4f}")